# M05D: Solutions — Cached Support Bot

Reference implementation for the Capstone #2 exercise.

**Design Pattern:**
1. **Inherit:** `CachedSupportBot` extends `ProductionSupportBot`
2. **Override:** Modify `chat()` to add cache checking
3. **Override:** Modify `stats()` to include cache metrics

---

## 🔧 Step 1: Setup

In [ ]:
import os
import hashlib
from pathlib import Path
from dotenv import load_dotenv

import openai
import tiktoken

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"

print(f"✅ Setup complete: Using {MODEL}!")

---

## 📦 Step 2: Base Components (from M05D Lab)

Included here so the solution is self-contained.

In [ ]:
class TokenCounter:
    """Count tokens for text."""
    
    def __init__(self, model=MODEL):
        try:
            self.tokenizer = tiktoken.encoding_for_model(model)
        except KeyError:
            try:
                self.tokenizer = tiktoken.get_encoding("o200k_base")
            except Exception:
                self.tokenizer = tiktoken.get_encoding("cl100k_base")
    
    def count(self, text):
        """Count tokens in text."""
        return len(self.tokenizer.encode(text))


PRICING = {
    "gpt-5": {"input": 1.25, "output": 10.00},
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60}
}


class CostTracker:
    """Track cumulative API costs with budget limits."""
    
    def __init__(self, model=MODEL, budget_limit=None):
        self.model = model
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.token_counter = TokenCounter(model)
        self.budget_limit = budget_limit
        self.request_count = 0
    
    def add_turn(self, user_message, assistant_message):
        """Record a turn and return its cost."""
        input_tokens = self.token_counter.count(user_message)
        output_tokens = self.token_counter.count(assistant_message)
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.request_count += 1
        return self._calculate_cost(input_tokens, output_tokens)
    
    def _calculate_cost(self, input_tokens, output_tokens):
        """Calculate cost for given token counts."""
        pricing = PRICING.get(self.model, PRICING["gpt-5-mini"])
        input_cost = (input_tokens / 1_000_000) * pricing["input"]
        output_cost = (output_tokens / 1_000_000) * pricing["output"]
        return input_cost + output_cost
    
    def get_total(self):
        """Get total cost so far."""
        return self._calculate_cost(self.total_input_tokens, self.total_output_tokens)
    
    def check_budget(self):
        """Check if within budget."""
        if self.budget_limit is None:
            return True
        return self.get_total() < self.budget_limit
    
    def budget_status(self):
        """Get budget status."""
        current_cost = self.get_total()
        usage_pct = (current_cost / self.budget_limit * 100) if self.budget_limit else 0
        return {
            "current_cost": current_cost,
            "budget_limit": self.budget_limit,
            "usage_percent": usage_pct,
            "tracked_events": self.request_count
        }


class ContextWindow:
    """Manage context with sliding window."""
    
    def __init__(self, max_turns=5):
        self.max_turns = max_turns
        self.message_history = []
    
    def add_message(self, speaker, text):
        """Add message and maintain window."""
        self.message_history.append({'speaker': speaker, 'text': text})
        max_messages = self.max_turns * 2
        if len(self.message_history) > max_messages:
            self.message_history[:] = self.message_history[-max_messages:]
    
    def get_messages(self):
        """Get current context."""
        return self.message_history.copy()
    
    def build_context(self):
        """Build context string for API calls."""
        context = ""
        for msg in self.message_history:
            context += f"{msg['speaker']}: {msg['text']}\n"
        return context


class ResponseCache:
    """Cache API responses to avoid duplicates."""
    
    def __init__(self):
        self.cache = {}
        self.hits = 0
        self.misses = 0
    
    def _make_key(self, prompt, instructions, model):
        """Create cache key from prompt + instructions + model."""
        text = f"{model}|{prompt.strip()}|{instructions.strip()}"
        return hashlib.md5(text.encode()).hexdigest()
    
    def get(self, prompt, instructions, model=MODEL):
        """Get cached response if exists."""
        key = self._make_key(prompt, instructions, model)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None
    
    def set(self, prompt, instructions, response, model=MODEL):
        """Store response in cache."""
        key = self._make_key(prompt, instructions, model)
        self.cache[key] = response
    
    def stats(self):
        """Get cache statistics."""
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total else 0
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": hit_rate,
            "size": len(self.cache)
        }


class ProductionSupportBot:
    """Production-ready customer support chatbot."""
    
    INSTRUCTIONS = "You are a customer support agent. Be concise."
    
    def __init__(self, client, max_turns=5, budget_limit=None, model=MODEL):
        self.client = client
        self.model = model
        self.context = ContextWindow(max_turns)
        self.cost_tracker = CostTracker(model, budget_limit)
        self.conversation_log = []
    
    def _generate_response(self, user_message):
        """Generate support response using local context."""
        response = self.client.responses.create(
            model=self.model,
            input=self.context.build_context(),
            instructions=self.INSTRUCTIONS
        )
        return response.output_text.strip()
    
    def chat(self, user_message):
        """Full pipeline: budget check → respond → track cost."""
        if not self.cost_tracker.check_budget():
            return {"response": "Budget exceeded."}
        
        self.context.add_message("user", user_message)
        assistant_message = self._generate_response(user_message)
        self.context.add_message("assistant", assistant_message)
        self.cost_tracker.add_turn(user_message, assistant_message)
        
        result = {"response": assistant_message}
        self.conversation_log.append(result)
        return result
    
    def stats(self):
        """Get session metrics."""
        return {
            "total_turns": len(self.conversation_log),
            "cost": self.cost_tracker.get_total(),
            "budget_status": self.cost_tracker.budget_status()
        }


# --------------------------------------------------------------
print("✅ Base components ready")

---

## 🤖 Step 3: Solution Implementation

Inherit from the base bot and add response caching.

In [ ]:
class CachedSupportBot(ProductionSupportBot):
    """Support bot with response caching."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.cache = ResponseCache()
    
    def chat(self, user_message):
        """Override to add caching."""
        if not self.cost_tracker.check_budget():
            return {"response": "Budget exceeded.", "cached": False}
        
        self.context.add_message("user", user_message)
        
        # Check cache for duplicate messages
        cached = self.cache.get(user_message, self.INSTRUCTIONS, self.model)
        if cached is not None:
            assistant_message = cached
            from_cache = True
        else:
            assistant_message = self._generate_response(user_message)
            self.cache.set(user_message, self.INSTRUCTIONS, assistant_message, self.model)
            from_cache = False
        
        self.context.add_message("assistant", assistant_message)
        self.cost_tracker.add_turn(user_message, assistant_message)
        
        result = {
            "response": assistant_message,
            "cached": from_cache
        }
        self.conversation_log.append(result)
        return result
    
    def stats(self):
        """Add cache metrics."""
        base_stats = super().stats()
        base_stats["cache"] = self.cache.stats()
        return base_stats


# --------------------------------------------------------------
print("✅ CachedSupportBot ready")

---

## 🎬 Demo: Cached Support Bot

In [ ]:
print("🤖 CACHED SUPPORT BOT DEMO")
print("="*60)

bot = CachedSupportBot(client, max_turns=5, budget_limit=0.50)

messages = [
    "I was charged twice for my subscription last month!",
    "How do I reset my password?",
    "I was charged twice for my subscription last month!",  # duplicate
    "Where is my order #12345?",
    "How do I reset my password?"  # duplicate
]

for turn_number, message in enumerate(messages, 1):
    print(f"\nTurn {turn_number}: {message}")
    result = bot.chat(message)
    source = "CACHE" if result["cached"] else "API"
    print(f"[{source:5}] {result['response'][:80]}{'...' if len(result['response']) > 80 else ''}")

print("\n" + "="*60)
print("SESSION STATS")
print("="*60)
final = bot.stats()
print(f"Total turns: {final['total_turns']}")
print(f"Total cost: ${final['cost']:.6f}")
print(f"Budget: {final['budget_status']['usage_percent']:.1f}% used")
print(f"Cache: {final['cache']['hits']} hits, {final['cache']['misses']} misses ({final['cache']['hit_rate']:.0f}% hit rate)")
print("="*60)

---

## 🎯 Key Takeaways

**Subclassing Pattern:**
- Extend `ProductionSupportBot` without modifying it
- Override `chat()` to add cache logic before and after the API call
- Call `super()` for `stats()` and extend with cache metrics

**Caching Integration:**
- Cache key = message + instructions + model
- Check cache before `_generate_response`, store after
- Track hit rate to measure savings

---